# AMD Hackathon — Vulnerability-Aware Java LLM

Script 1b: Fetch actual Java source code for CWE-Bench-Java from GitHub

CWE-Bench-Java (iris-sast) provides CVE/CWE metadata + GitHub commit references,
but NOT the actual code. This script reads project_info.csv + fix_info.csv,
then fetches the vulnerable (before) and fixed (after) Java code via the
GitHub API and saves it to raw_data/cwe_bench_java/code/.

Usage:
  pip install requests
  export GITHUB_TOKEN=ghp_your_token_here    # recommended — 60 req/hr without token
  python 01b_fetch_cwe_bench_code.py

  Optional:
    --max-entries 50     limit number of CVEs to process (default: all)
    --workers 4          parallel worker threads (default: 4)

In [ ]:
from __future__ import annotations

import csv
import json
import os
import re
import sys
import time
import argparse
import urllib.request
import urllib.error
import ssl
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional

RAW_DIR  = Path("raw_data")
DATA_DIR = RAW_DIR / "cwe_bench_java"
CODE_DIR = DATA_DIR / "code"

GITHUB_API = "https://api.github.com"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

## Helpers

In [ ]:
def _ssl_ctx():
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    return ctx

In [ ]:
def github_get(url: str, retries: int = 3) -> Optional[dict | list | str]:
    """Call GitHub API with optional auth token, returns parsed JSON or None."""
    headers = {
        "User-Agent": "amd-hackathon-dataset-fetcher/1.0",
        "Accept": "application/vnd.github.v3+json",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"token {GITHUB_TOKEN}"

    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30, context=_ssl_ctx()) as r:
                # Check rate limit
                remaining = int(r.headers.get("X-RateLimit-Remaining", 999))
                if remaining < 5:
                    reset_ts = int(r.headers.get("X-RateLimit-Reset", 0))
                    wait = max(1, reset_ts - time.time() + 5)
                    print(f"  [RATE LIMIT] Waiting {wait:.0f}s for reset...")
                    time.sleep(wait)
                content = r.read().decode("utf-8", errors="ignore")
                return json.loads(content) if content.startswith(("{", "[")) else content
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            if e.code == 403:
                print(f"  [403] Rate limited or auth needed. Set GITHUB_TOKEN env var.")
                time.sleep(60)
            elif e.code == 422:
                return None
            else:
                print(f"  [HTTP {e.code}] {url}")
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  [ERROR] {e}: {url}")
            time.sleep(2 ** attempt)
    return None

In [ ]:
def get_file_at_commit(
    owner: str, repo: str, commit_sha: str, file_path: str
) -> Optional[str]:
    """Fetch raw content of a file at a specific commit."""
    url = f"{GITHUB_API}/repos/{owner}/{repo}/contents/{file_path}?ref={commit_sha}"
    data = github_get(url)
    if not data or not isinstance(data, dict):
        return None

    import base64
    content_b64 = data.get("content", "")
    if content_b64:
        try:
            return base64.b64decode(content_b64.replace("\n", "")).decode("utf-8", errors="ignore")
        except Exception:
            return None
    return None

In [ ]:
def get_parent_commit(owner: str, repo: str, commit_sha: str) -> Optional[str]:
    """Return the first parent SHA of a commit."""
    url = f"{GITHUB_API}/repos/{owner}/{repo}/commits/{commit_sha}"
    data = github_get(url)
    if data and isinstance(data, dict):
        parents = data.get("parents", [])
        if parents:
            return parents[0]["sha"]
    return None

In [ ]:
def sanitize_filename(s: str) -> str:
    """Make a string safe for use as a filename."""
    return re.sub(r'[^a-zA-Z0-9_\-]', '_', s)

## Core fetch logic

In [ ]:
def fetch_one_entry(entry: dict) -> tuple[str, bool, str]:
    """
    Fetch before/after code for one fix_info entry.
    Returns (entry_id, success, message).
    """
    cve_id   = entry["cve_id"]
    owner    = entry["github_username"]
    repo     = entry["github_repository_name"]
    commit   = entry["commit"]       # fix commit
    file_p   = entry["file"]
    method   = entry["method"]
    slug     = entry["project_slug"]

    # Safe filename prefix
    prefix = sanitize_filename(f"{slug}_{method}")
    before_path = CODE_DIR / f"{prefix}_before.java"
    after_path  = CODE_DIR / f"{prefix}_after.java"
    meta_path   = CODE_DIR / f"{prefix}_meta.json"

    # Skip if already fetched
    if before_path.exists() and after_path.exists():
        return (prefix, True, "already exists")

    # Get the "after" code (at fix commit)
    after_code = get_file_at_commit(owner, repo, commit, file_p)
    if not after_code:
        return (prefix, False, f"Could not fetch fixed code for {file_p}@{commit[:8]}")

    # Get the parent commit (for "before" code)
    parent_sha = get_parent_commit(owner, repo, commit)
    if not parent_sha:
        return (prefix, False, f"Could not find parent commit of {commit[:8]}")

    before_code = get_file_at_commit(owner, repo, parent_sha, file_p)
    if not before_code:
        return (prefix, False, f"Could not fetch vulnerable code for {file_p}@{parent_sha[:8]}")

    # Write files
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    before_path.write_text(before_code, encoding="utf-8")
    after_path.write_text(after_code, encoding="utf-8")
    meta_path.write_text(
        json.dumps({
            "cve_id":      cve_id,
            "project_slug": slug,
            "owner":       owner,
            "repo":        repo,
            "commit_hash": commit,
            "parent_hash": parent_sha,
            "file_path":   file_p,
            "method_name": method,
        }, indent=2),
        encoding="utf-8",
    )
    return (prefix, True, f"OK ({len(before_code)}/{len(after_code)} chars)")

## Main

In [ ]:
# ── Configuration — edit these variables before running ───────────────────────
import os

# GitHub token for higher API rate limits (5000 req/hr vs 60 unauthenticated)
# Set here or via environment variable: export GITHUB_TOKEN=ghp_xxxx
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")  # or: GITHUB_TOKEN = "ghp_xxxx"

# Max number of CVE entries to fetch. None = fetch all (~213 entries).
MAX_ENTRIES = None  # e.g. 50 for a quick test

# Number of parallel worker threads (max recommended: 8)
WORKERS = 4

In [ ]:
# ── Run fetch ──────────────────────────────────────────────────────────────────
if not GITHUB_TOKEN:
    print("[WARN] GITHUB_TOKEN not set. GitHub API rate limit: 60 req/hr (unauthenticated).")
    print("       Set GITHUB_TOKEN = 'ghp_xxx' in the Configuration cell for 5000 req/hr.")

fix_csv = DATA_DIR / "fix_info.csv"
if not fix_csv.exists():
    print(f"[ERROR] {fix_csv} not found. Run the 01_download_datasets notebook first.")
else:
    entries = []
    with open(fix_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("file", "").endswith(".java"):
                entries.append(dict(row))

    if MAX_ENTRIES:
        entries = entries[:MAX_ENTRIES]

    print(f"[INFO] Fetching code for {len(entries)} entries with {WORKERS} workers...")
    CODE_DIR.mkdir(parents=True, exist_ok=True)

    success = fail = 0
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futures = {pool.submit(fetch_one_entry, e): e for e in entries}
        for fut in as_completed(futures):
            prefix, ok, msg = fut.result()
            if ok:
                success += 1
                print(f"  [OK]  {prefix}: {msg}")
            else:
                fail += 1
                print(f"  [FAIL] {prefix}: {msg}")

    print(f"\n✓ Done: {success} fetched, {fail} failed.")
    print(f"  Code saved to: {CODE_DIR.absolute()}")
    print("  Next step: run the 02_clean_and_convert notebook")